# IDP Metadata Collector Framework
This Databricks notebook wires the production-ready collector module into a runnable job with widgets for toggling full loads, row counts, and additional options.

In [ ]:
# COMMAND ----------
import uuid

full_load_default = "False"
compute_row_count_default = "False"
include_inactive_default = "False"

if 'dbutils' in globals():
    dbutils.widgets.text("full_load", full_load_default, "Full Load?")
    dbutils.widgets.text("compute_row_count", compute_row_count_default, "Compute Row Count?")
    dbutils.widgets.text("include_inactive", include_inactive_default, "Process Inactive Sources?")
    dbutils.widgets.text("job_run_id", str(uuid.uuid4()), "Job Run ID")

In [ ]:
# COMMAND ----------
def _widget_bool(name: str, default: str = "False") -> bool:
    if 'dbutils' not in globals():
        return default.lower() in {"true", "1", "yes"}
    return dbutils.widgets.get(name).strip().lower() in {"true", "1", "yes"}

full_load = _widget_bool("full_load", "False")
compute_row_count = _widget_bool("compute_row_count", "False")
include_inactive = _widget_bool("include_inactive", "False")
job_run_id = dbutils.widgets.get("job_run_id") if 'dbutils' in globals() else str(uuid.uuid4())

In [ ]:
# COMMAND ----------
from IDP_Metadata_Collector_Framework import (
    CollectorContext,
    CollectorFactory,
    ConfigRepository,
    MetadataCollectorOrchestrator,
    DatabricksSecretProvider,
    DbutilsFileSystemClient,
    run_job,
)

secret_provider = DatabricksSecretProvider(dbutils) if 'dbutils' in globals() else None
filesystem_client = DbutilsFileSystemClient(dbutils) if 'dbutils' in globals() else None

In [ ]:
# COMMAND ----------
if secret_provider is None:
    raise RuntimeError("Databricks secret provider is required inside this notebook")

run_job(
    spark=spark,
    secret_provider=secret_provider,
    filesystem_client=filesystem_client,
    compute_row_count=compute_row_count,
    full_load=full_load,
    include_inactive=include_inactive,
)

print(f"Run {job_run_id} completed")

In [ ]:
# COMMAND ----------
summary_table = "qa_idp.logs.meta_data_registry_summary"
try:
    summary_df = spark.table(summary_table).orderBy("source_id")
    display(summary_df)
except Exception as exc:
    print(f"Summary table {summary_table} not yet available: {exc}")